In [ ]:
import datajoint as dj
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sb
import statistics as stat
import random
from scipy.ndimage import gaussian_filter1d
import matplotlib.cm as cm  # For colormaps
import matplotlib.gridspec as gridspec

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import datajoint as dj
import os
import numpy as np
import statistics as stat

dj.config['database.host'] = "arseny-lab.cmte3q4ziyvy.il-central-1.rds.amazonaws.com"
dj.config['database.user'] = "talch012"
dj.config['database.password'] = "simple"

In [ ]:
imgn= dj.schema('talch012_imgt')
imgn= dj.VirtualModule('IMGt', 'talch012_imgt')
lickiti= dj.schema('talch012_lickiti')
lickiti= dj.VirtualModule ('LICKITI', 'talch012_lickiti')
EXPn= dj.schema('talch012_expt')
EXPn= dj.VirtualModule ('EXPt', 'talch012_expt')
lab= dj.schema('talch012_labt')
lab= dj.VirtualModule ('LABt', 'talch012_labt')


In [4]:
# pure random trials [-10,20]
img= dj.schema('talch012_imgt')
img= dj.VirtualModule('IMGt', 'talch012_imgt')
lickiti= dj.schema('talch012_lickiti')
lickiti= dj.VirtualModule ('LICKITI', 'talch012_lickiti')
exp= dj.schema('talch012_expt')
exp= dj.VirtualModule ('EXPt', 'talch012_expt')


In [ ]:
# good works smoothing and stability calculation code
import datajoint as dj
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr

# Import schemas as VirtualModule to access their tables
img = dj.VirtualModule('img', 'talch012_imgt')
lickiti = dj.VirtualModule('lickiti', 'talch012_lickiti')
exp = dj.VirtualModule('exp', 'talch012_expt')

# Create schema for new tables
schema = dj.schema('talch012_lickiti')

@schema
class ROILickITILongPositionSmoothed(dj.Imported):
    definition = """
    # Smoothed PSTH traces using Gaussian or rolling window
    -> exp.SessionEpoch
    -> img.ROI
    smoothing_method        : varchar(32)   # 'gaussian' or 'rolling_window'
    sigma                   : float         # sigma for gaussian or window size for rolling
    ---
    psth_regular_long_right             : longblob
    psth_regular_odd_long_right         : longblob
    psth_regular_even_long_right        : longblob
    psth_small_long_right=null          : longblob
    psth_small_odd_long_right=null      : longblob
    psth_small_even_long_right=null     : longblob
    psth_large_long_right=null          : longblob
    psth_large_odd_long_right=null      : longblob
    psth_large_even_long_right=null     : longblob
    psth_regular_stem_long_right        : longblob
    psth_small_stem_long_right=null     : longblob
    psth_large_stem_long_right=null     : longblob
    psth_regular_long_left              : longblob
    psth_regular_odd_long_left          : longblob
    psth_regular_even_long_left         : longblob
    psth_small_long_left=null           : longblob
    psth_small_odd_long_left=null       : longblob
    psth_small_even_long_left=null      : longblob
    psth_large_long_left=null           : longblob
    psth_large_odd_long_left=null       : longblob
    psth_large_even_long_left=null      : longblob
    psth_regular_stem_long_left         : longblob
    psth_small_stem_long_left=null      : longblob
    psth_large_stem_long_left=null      : longblob
    psth_time                           : longblob
    """
    
    # Key source: sessions with PSTH data
    key_source = exp.SessionEpoch & lickiti.ROILickITILongPosition & 'session_epoch_type="behav_only"'
    
    def make(self, key):
        """Populate smoothed PSTH for all ROIs in a session"""
        
        # Smoothing parameters
        smoothing_method = 'gaussian'
        sigma = 10
        
        # Fetch original PSTH data with joins
        try:
            orig_data = (lickiti.ROILickITILongPosition * img.ROIuid * img.CellRegtoROI & key).fetch(order_by='roi_number')
        except:
            # Fallback without joins if ROIuid/CellRegtoROI not available
            orig_data = (lickiti.ROILickITILongPosition & key).fetch(order_by='roi_number')
        
        if len(orig_data) == 0:
            print(f"No PSTH data found for {key}, skipping...")
            return
        
        # Get time vector
        psth_time_raw = orig_data[0]['psth_time']
        if psth_time_raw is None or len(psth_time_raw) == 0:
            print(f"No psth_time found for {key}, skipping...")
            return
        
        # Handle nested array format
        psth_time = np.array(psth_time_raw).squeeze()
        if psth_time.ndim == 0:
            print(f"Invalid psth_time for {key}, skipping...")
            return
        m = len(psth_time)
        
        n = len(orig_data)
        print(f"Processing {n} ROIs for session {key}")
        
        # Define PSTH fields
        reward_types = ['regular', 'large', 'small']
        positions = ['right', 'left']
        suffixes = ['', '_odd', '_even']
        
        psth_field_names = []
        for reward in reward_types:
            for position in positions:
                for suffix in suffixes:
                    psth_field_names.append(f"psth_{reward}{suffix}_long_{position}")
        
        # Preallocate arrays
        psth_arrays = {field: np.zeros((n, m)) for field in psth_field_names}
        
        # Fill arrays
        for i, entry in enumerate(orig_data):
            for reward in reward_types:
                for position in positions:
                    for suffix in suffixes:
                        key_name = f"psth_{reward}{suffix}_long_{position}"
                        
                        # Check field exists
                        try:
                            data = entry[key_name]
                        except (ValueError, KeyError):
                            continue
                        
                        if data is not None:
                            arr = np.array(data).squeeze()
                            if arr.size == 0 or arr.ndim == 0:
                                continue
                            
                            # Handle length mismatch
                            if len(arr) > m:
                                arr = arr[:m]
                            elif len(arr) < m:
                                m = len(arr)
                                for k in psth_arrays:
                                    psth_arrays[k] = psth_arrays[k][:, :m]
                                arr = arr[:m]
                            
                            psth_arrays[key_name][i, :] = arr
        
        # Smooth all arrays
        if smoothing_method == 'gaussian':
            for field_name in psth_field_names:
                psth_arrays[field_name] = gaussian_filter1d(psth_arrays[field_name], sigma=sigma, axis=1)
        
        # Update time vector length
        psth_time = psth_time[:m]
        
        # Insert in batches
        batch_size = 500
        batch = []
        
        for i, entry in enumerate(orig_data):

            tuple_entry = {
                'subject_id': entry['subject_id'],
                'session': entry['session'],
                'session_epoch_type': entry['session_epoch_type'],
                'session_epoch_number': entry['session_epoch_number'],
                'plane_num': entry['plane_num'],
                'channel_num': entry['channel_num'],
                'fov_num': entry['fov_num'],
                'roi_number': entry['roi_number'],
                'smoothing_method': smoothing_method,
                'sigma': sigma
            }
            # Add smoothed PSTH data
            for field_name in psth_field_names:
                tuple_entry[field_name] = psth_arrays[field_name][i, :]
            
            # Add stem fields (unsmoothed)
            stem_fields = ['psth_regular_stem_long_right', 'psth_small_stem_long_right', 
                          'psth_large_stem_long_right', 'psth_regular_stem_long_left',
                          'psth_small_stem_long_left', 'psth_large_stem_long_left']
            
            for field_name in stem_fields:
                try:
                    data = entry[field_name]
                    if data is not None:
                        arr = np.array(data).squeeze()
                        if arr.ndim > 0 and len(arr) > m:
                            arr = arr[:m]
                        tuple_entry[field_name] = arr
                    else:
                        tuple_entry[field_name] = None
                except (ValueError, KeyError):
                    tuple_entry[field_name] = None
            
            # Add time vector
            tuple_entry['psth_time'] = psth_time
            
            batch.append(tuple_entry)
            
            # Insert batch
            if len(batch) >= batch_size or i == n - 1:
                if len(batch) > 0:
                    try:
                        self.insert(batch, skip_duplicates=True)
                        print(f"  Inserted {min(i+1, n)}/{n} ROIs")
                    except Exception as e:
                        print(f"  Error inserting batch: {e}")
                        for entry in batch:
                            try:
                                self.insert1(entry, skip_duplicates=True)
                            except Exception as e2:
                                print(f"  Failed ROI {entry['roi_number']}: {e2}")
                    batch = []


@schema
class ROIStability(dj.Imported):
    definition = """
    # Odd-even trial stability (correlation) for different time intervals
    -> exp.SessionEpoch
    -> img.ROI
    time_interval_response  : varchar(32)   # e.g., '[0, 2]'
    time_interval_delay     : varchar(32)   # e.g., '[4, 10]'
    time_interval_full      : varchar(32)   # e.g., '[-5, 10]'
    ---
    stability_regular_right_response    : float
    stability_regular_left_response     : float
    stability_regular_side_response     : float  # right and left concatenated
    stability_large_right_response      : float
    stability_large_left_response       : float
    stability_large_response            : float  # right and left concatenated
    stability_regular_right_delay       : float
    stability_regular_left_delay        : float
    stability_regular_side_delay        : float
    stability_large_right_delay         : float
    stability_large_left_delay          : float
    stability_large_delay               : float
    stability_regular_right_full        : float
    stability_regular_left_full         : float
    stability_regular_side_full         : float
    stability_large_right_full          : float
    stability_large_left_full           : float
    stability_large_full                : float
    """
    
    # Key source: sessions with PSTH data
    key_source = exp.SessionEpoch & lickiti.ROILickITILongPosition & 'session_epoch_type="behav_only"'
    
    def make(self, key):
        """Calculate stability metrics for all ROIs in a session"""
        
        # Time intervals
        time_interval_response = '[0, 2]'
        time_interval_delay = '[4, 10]'
        time_interval_full = '[-5, 10]'
        
        # Fetch PSTH data
        try:
            psth_data = (lickiti.ROILickITILongPosition * img.ROIuid * img.CellRegtoROI & key).fetch(order_by='roi_number')
        except:
            psth_data = (lickiti.ROILickITILongPosition & key).fetch(order_by='roi_number')
        
        if len(psth_data) == 0:
            print(f"No PSTH data found for {key}, skipping...")
            return
        
        # Get time vector
        psth_time = np.array(psth_data[0]['psth_time']).squeeze()
        
        # Define time indices
        resp_idx = np.where((psth_time >= 0) & (psth_time <= 2))[0]
        delay_idx = np.where((psth_time >= 4) & (psth_time <= 10))[0]
        full_idx = np.where((psth_time >= -5) & (psth_time <= 10))[0]
        
        print(f"Processing stability for {len(psth_data)} ROIs in session {key}")
        
        # Process in batches
        batch_size = 500
        batch = []
        n = len(psth_data)
        
        for i, row in enumerate(psth_data):
            tuple_entry = {
                'subject_id': int(row['subject_id']),
                'session': int(row['session']),
                'session_epoch_type': row['session_epoch_type'],
                'session_epoch_number': int(row['session_epoch_number']),
                'plane_num': int(row['plane_num']),
                'channel_num': int(row['channel_num']),
                'fov_num': int(row['fov_num']),
                'roi_number': int(row['roi_number']),
                'time_interval_response': time_interval_response,
                'time_interval_delay': time_interval_delay,
                'time_interval_full': time_interval_full
            }

            # Calculate stability for each interval
            intervals = [resp_idx, delay_idx, full_idx]
            interval_names = ['response', 'delay', 'full']
            
            for idx, int_name in zip(intervals, interval_names):
                # Regular right
                tuple_entry[f'stability_regular_right_{int_name}'] = \
                    compute_odd_even_corr(row, 'psth_regular_odd_long_right', 
                                         'psth_regular_even_long_right', idx)
                
                # Regular left
                tuple_entry[f'stability_regular_left_{int_name}'] = \
                    compute_odd_even_corr(row, 'psth_regular_odd_long_left', 
                                         'psth_regular_even_long_left', idx)
                
                # Regular side (concatenated)
                tuple_entry[f'stability_regular_side_{int_name}'] = \
                    compute_concat_odd_even_corr(row, 
                        'psth_regular_odd_long_right', 'psth_regular_odd_long_left',
                        'psth_regular_even_long_right', 'psth_regular_even_long_left', idx)
                
                # Large right
                tuple_entry[f'stability_large_right_{int_name}'] = \
                    compute_odd_even_corr(row, 'psth_large_odd_long_right', 
                                         'psth_large_even_long_right', idx)
                
                # Large left
                tuple_entry[f'stability_large_left_{int_name}'] = \
                    compute_odd_even_corr(row, 'psth_large_odd_long_left', 
                                         'psth_large_even_long_left', idx)
                
                # Large (concatenated)
                tuple_entry[f'stability_large_{int_name}'] = \
                    compute_concat_odd_even_corr(row,
                        'psth_large_odd_long_right', 'psth_large_odd_long_left',
                        'psth_large_even_long_right', 'psth_large_even_long_left', idx)
            
            batch.append(tuple_entry)
            
            # Insert batch
            if len(batch) >= batch_size or i == n - 1:
                if len(batch) > 0:
                    try:
                        self.insert(batch, skip_duplicates=True)
                        print(f"  Inserted {min(i+1, n)}/{n} ROIs")
                    except Exception as e:
                        print(f"  Error inserting batch: {e}")
                        for entry in batch:
                            try:
                                self.insert1(entry, skip_duplicates=True)
                            except Exception as e2:
                                print(f"  Failed ROI {entry['roi_number']}: {e2}")
                    batch = []


# Helper functions
def compute_odd_even_corr(row, odd_field, even_field, idx):
    """Compute correlation between odd and even trials"""
    try:
        odd_data = row[odd_field]
        even_data = row[even_field]
        
        if odd_data is None or even_data is None:
            return np.nan
        
        odd_data = np.array(odd_data).squeeze()
        even_data = np.array(even_data).squeeze()
        
        if odd_data.size == 0 or even_data.size == 0:
            return np.nan
        
        odd_interval = odd_data[idx]
        even_interval = even_data[idx]
        
        if np.any(odd_interval) and np.any(even_interval):
            corr, _ = pearsonr(odd_interval, even_interval)
            return float(corr)
        else:
            return np.nan
    except Exception as e:
        return np.nan


def compute_concat_odd_even_corr(row, odd_right_field, odd_left_field, even_right_field, even_left_field, idx):
    """Compute correlation between concatenated odd and even trials"""
    try:
        odd_right = row[odd_right_field]
        odd_left = row[odd_left_field]
        even_right = row[even_right_field]
        even_left = row[even_left_field]
        
        if any(x is None for x in [odd_right, odd_left, even_right, even_left]):
            return np.nan
        
        odd_right = np.array(odd_right).squeeze()
        odd_left = np.array(odd_left).squeeze()
        even_right = np.array(even_right).squeeze()
        even_left = np.array(even_left).squeeze()
        
        if any(x.size == 0 for x in [odd_right, odd_left, even_right, even_left]):
            return np.nan
        
        odd_concat = np.concatenate([odd_right[idx], odd_left[idx]])
        even_concat = np.concatenate([even_right[idx], even_left[idx]])
        
        if np.any(odd_concat) and np.any(even_concat):
            corr, _ = pearsonr(odd_concat, even_concat)
            return float(corr)
        else:
            return np.nan
    except Exception as e:
        return np.nan


# === HOW TO USE ===
#
# 1. Populate tables:
#    ROILickITILongPositionSmoothed().populate(display_progress=True)
#    ROIStability().populate(display_progress=True)
#
# 2. Retrieve data (same as original table):
#    smoothed_long = (lickiti.ROILickITILongPositionSmoothed & 
#                     'smoothing_method="gaussian"' & 'sigma=10').fetch(order_by='roi_number')
#    
#    # Access like original:
#    psth_time = smoothed_long[0]['psth_time']
#    right_trace = smoothed_long[i]['psth_regular_long_right']
#    
# 3. Get stability:
#    stability = lickiti.ROIStability.fetch(order_by='roi_number')
#    response_stability = stability[i]['stability_regular_side_response']

In [44]:
ROILickITILongPositionSmoothed.populate(display_progress=True)

ROILickITILongPositionSmoothed:  25%|██▌       | 1/4 [00:00<00:02,  1.11it/s]

No PSTH data found for {'subject_id': 101103, 'session': 5, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}, skipping...


ROILickITILongPositionSmoothed:  50%|█████     | 2/4 [00:01<00:01,  1.71it/s]

No PSTH data found for {'subject_id': 101106, 'session': 2, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}, skipping...


ROILickITILongPositionSmoothed:  75%|███████▌  | 3/4 [00:01<00:00,  2.07it/s]

No PSTH data found for {'subject_id': 101106, 'session': 5, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}, skipping...


ROILickITILongPositionSmoothed: 100%|██████████| 4/4 [00:01<00:00,  2.01it/s]

No PSTH data found for {'subject_id': 101106, 'session': 6, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}, skipping...


In [65]:
ROIStability.populate(display_progress=True)

ROIStability:   0%|          | 0/13 [00:00<?, ?it/s]

Processing stability for 1647 ROIs in session {'subject_id': 101102, 'session': 16, 'session_epoch_type': 'behav_only', 'session_epoch_number': 1}
  Inserted 500/1647 ROIs


c:\Users\admin\anaconda3.1\envs\TalA_fixed\lib\site-packages\scipy\stats\_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


  Error inserting batch: Field 'stability_regular_right_response' doesn't have a default value
  Failed ROI 882: Field 'stability_regular_right_response' doesn't have a default value
  Inserted 1500/1647 ROIs


ROIStability:   8%|▊         | 1/13 [01:26<17:22, 86.87s/it]

...

No PSTH data found for {'subject_id': 101106, 'session': 5, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}, skipping...


ROIStability:  92%|█████████▏| 12/13 [12:06<00:22, 22.67s/it]

No PSTH data found for {'subject_id': 101106, 'session': 6, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}, skipping...
Processing stability for 329 ROIs in session {'subject_id': 101106, 'session': 7, 'session_epoch_type': 'behav_only', 'session_epoch_number': 2}


ROIStability: 100%|██████████| 13/13 [12:43<00:00, 58.74s/it]

  Inserted 329/329 ROIs


In [68]:
ROIStability&'subject_id=101106'

subject_id institution 6 digit animal ID,session session number,session_epoch_type,session_epoch_number session epoch number,fov_num field of view number (restarts for every session),plane_num imaging plane within this field of view,"channel_num imaging channel (e.g. 1 green, 2 red etc)",roi_number roi number (restarts for every session),"time_interval_response e.g., '[0, 2]'","time_interval_delay e.g., '[4, 10]'","time_interval_full e.g., '[-5, 10]'",stability_regular_right_response,stability_regular_left_response,stability_regular_side_response right and left concatenated,stability_large_right_response,stability_large_left_response,stability_large_response right and left concatenated,stability_regular_right_delay,stability_regular_left_delay,stability_regular_side_delay,stability_large_right_delay,stability_large_left_delay,stability_large_delay,stability_regular_right_full,stability_regular_left_full,stability_regular_side_full,stability_large_right_full,stability_large_left_full,stability_large_full
101106,3,behav_only,2,1,1,1,1,"[0, 2]","[4, 10]","[-5, 10]",0.980241,0.917752,0.954155,0.702165,0.670888,0.684946,0.404508,-0.136563,0.162757,0.0101217,0.526424,0.187279,0.883375,0.749096,0.820352,-0.235637,0.389218,-0.0192923
101106,3,behav_only,2,1,1,1,2,"[0, 2]","[4, 10]","[-5, 10]",0.84725,0.894266,0.866261,0.418147,0.406319,0.321979,0.176402,-0.0966387,0.0948497,-0.102364,0.607591,0.021481,0.5341,0.694156,0.610997,-0.166362,0.170186,0.0120904
101106,3,behav_only,2,1,1,1,3,"[0, 2]","[4, 10]","[-5, 10]",0.974594,0.967036,0.959637,0.948716,0.894084,0.861605,0.555619,-0.209303,-0.316623,-0.105137,0.133353,0.151381,0.928604,0.735271,0.813553,0.656637,0.720327,0.640494
101106,3,behav_only,2,1,1,1,4,"[0, 2]","[4, 10]","[-5, 10]",0.973892,0.962413,0.87262,0.910749,0.934988,0.818449,-0.241048,-0.288349,0.216617,-0.428512,0.00439466,-0.115936,0.927715,0.916628,0.90006,0.632572,0.854169,0.688763
101106,3,behav_only,2,1,1,1,5,"[0, 2]","[4, 10]","[-5, 10]",0.627355,-0.0395808,0.331886,0.529238,0.0919861,0.0333886,0.112601,-0.0946659,-0.0236981,-0.0151596,-0.293041,-0.168376,0.467958,0.571871,0.520363,0.412431,0.199859,0.257651
101106,3,behav_only,2,1,1,1,6,"[0, 2]","[4, 10]","[-5, 10]",0.910212,0.918133,0.823168,0.545586,0.108545,0.373416,0.223122,0.00277504,0.0440575,-0.157704,0.290863,0.119321,0.723883,0.724917,0.700258,0.268164,0.558057,0.44352
101106,3,behav_only,2,1,1,1,7,"[0, 2]","[4, 10]","[-5, 10]",0.893514,0.81569,0.784922,0.0598069,0.791027,0.392324,0.0829521,0.389217,0.299156,0.0627202,-0.305728,-0.190765,0.863938,0.858494,0.852924,0.599171,0.449393,0.548705
101106,3,behav_only,2,1,1,1,8,"[0, 2]","[4, 10]","[-5, 10]",0.948225,0.611561,0.830624,-0.332546,-0.406267,-0.234345,0.609896,0.058458,0.40945,0.109477,-0.21989,-0.144621,0.915577,0.841934,0.87376,0.597747,0.250738,0.379345
101106,3,behav_only,2,1,1,1,9,"[0, 2]","[4, 10]","[-5, 10]",0.966679,0.938064,0.919066,0.797369,0.817962,0.668118,0.135252,0.710017,0.380828,0.0805216,0.355287,-0.156146,0.907934,0.766989,0.770968,0.409591,0.457068,0.23875
101106,3,behav_only,2,1,1,1,10,"[0, 2]","[4, 10]","[-5, 10]",0.845316,0.253873,-0.00131554,-0.612403,0.614524,0.209105,0.655754,-0.289129,-0.169491,-0.325573,-0.0456579,-0.14352,0.990718,0.92076,0.931595,0.851813,0.907818,0.87388


In [ ]:
# ## retrival
# smoothed_long = (ROILickITILongPositionSmoothed  & 
#                  'smoothing_method="gaussian"' & 'sigma=10').fetch(order_by='roi_number')

# # Access exactly like your working script:
# psth_time = smoothed_long[0]['psth_time']  
# right_trace = smoothed_long[10]['psth_regular_long_right']  

# # Then use in loops exactly like your working code
# for i, entry in enumerate(smoothed_long):
#     arr = np.array(entry['psth_regular_long_right']).squeeze()
#     # Now arr is 1D array ready to use

In [ ]:
#Noa's raw stat hist + stability+ peak psth+ wideth 
import datajoint as dj
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from scipy.ndimage import gaussian_filter1d

# --- Attach schema ---
schema = dj.Schema("talch012_plots")


@schema
class RawStats(dj.Computed):
    definition = """
    -> lab.Subject
    ---
    raw_stats_path : varchar(255)  # full path to saved figure
    """

    def make(self, key):
        print(f"Processing subject {key['subject_id']}...")

        # ---------- Relations ----------
        rel_ROI = (imgn.ROIuid & imgn.ROIGood - imgn.ROIBad) * imgn.ROIPositionETL * imgn.FOVEpoch
        fpicks = imgn.ROIFTracePeakStats.proj(
            'pure_f_maximum_val',
            'median_f_maximum_val',
            'std_f_maximum_val',
        )
        dffpicks = imgn.ROIdeltaFPeakStats.proj(
            'pure_maximum_val',
            'median_maximum_val',
            'std_maximum_val',
        )
        combined = (
            EXPn.SessionBehavioral * rel_ROI *
            imgn.ROIFMean * imgn.ROIdeltaFMean *
            fpicks * dffpicks * lickiti.ROIITIlongDeltaFPureConsecutiveTrials
        ) & key

        psth_data = combined.fetch(format='frame')
        if psth_data.empty:
            print(f"No PSTH data for subject {key['subject_id']}. Skipping.")
            self.insert1({**key, 'raw_stats_path': 'None'})
            return

        plane = (imgn.Plane & key).fetch(format='frame')

        # ---------- Fetch stability data ----------
        midposition = (lickiti.ROILickITIMidPosition * imgn.ROIuid * imgn.CellRegtoROI & key).fetch(order_by='roi_number')
        short_position = (lickiti.ROILickITIShortPosition * imgn.ROIuid * imgn.CellRegtoROI & key).fetch(order_by='roi_number')
        long_position = (lickiti.ROILickITILongPosition * imgn.ROIuid * imgn.CellRegtoROI & key).fetch(order_by='roi_number')
        
        if len(long_position) == 0:
            print(f"No stability data for subject {key['subject_id']}. Proceeding without stability.")
            stability_data = None
        else:
            psth_time = long_position[0]['psth_time'][0]
            stability_data = self._compute_stability(long_position, midposition, short_position, psth_time)

        # ---------- Helper functions ----------
        def _peak_psth_series(df):
            """Get maximum value from each PSTH"""
            vals = df['psth_regular_long'].values
            return np.array([
                np.asarray(v[0]).max() if hasattr(v, '__len__') and len(v[0]) > 0 else np.nan
                for v in vals
            ], dtype=float)

        def _darken_color(color, factor=0.8):
            import matplotlib.colors as mcolors
            rgba = mcolors.to_rgba(color)
            return (rgba[0]*factor, rgba[1]*factor, rgba[2]*factor, rgba[3])

        def crop_sides(img, crop=dict(top=150, bottom=50, left=100, right=100)):
            h, w = img.shape[:2]
            return img[crop['top']: h - crop['bottom'],
                       crop['left']: w - crop['right']]

        # ---------- Plot config ----------
        INCLUDE_FOV = True
        WIDTH_PER_COL = 5.5
        NBINS_MAIN = 120
        OVERFLOW_Q = 0.99
        OVERFLOW_DARKEN = 0.75
        FORCE_ZERO_XLIM_COLS = {0, 2, 3, 4}

        session_numbers = psth_data.index.get_level_values('session').unique()

        metrics = [
            dict(key='mean_dff', title=r'$\Delta F/F$ Mean', color='lightblue',
                 xlabel=r'$\Delta F/F$', global_values=psth_data['mean_dff'].values),
            dict(key='f_mean', title='F Mean', color='steelblue',
                 xlabel='F', global_values=psth_data['f_mean'].values),
            dict(key='__peak_psth__', title='Peak PSTH', color='green',
                 xlabel='PSTH (peak)', global_values=_peak_psth_series(psth_data)),
        ]

        # Only include pure and median, skip average
        for col, title, color in [
            ('pure_maximum_val', 'Pure ΔF/F max', 'tab:orange'),
            ('median_maximum_val', 'Median top-10 ΔF/F', 'darkorange'),
        ]:
            if col in psth_data.columns:
                metrics.append(dict(key=col, title=title, color=color,
                                    xlabel=title, global_values=psth_data[col].values))

        for col, title, color in [
            ('pure_f_maximum_val', 'Pure F max', 'mediumpurple'),
            ('median_f_maximum_val', 'Median top-10 F', 'indigo'),
        ]:
            if col in psth_data.columns:
                metrics.append(dict(key=col, title=title, color=color,
                                    xlabel=title, global_values=psth_data[col].values))

        # ---------- Global binning ----------
        for m in metrics:
            x = np.asarray(m['global_values'])
            x = x[np.isfinite(x)]
            if x.size == 0:
                m['lo'], m['cap'], m['binw'] = 0.0, 1.0, 1.0 / NBINS_MAIN
                m['edges_main'] = np.linspace(m['lo'], m['cap'], NBINS_MAIN + 1)
                continue
            lo = np.min(x)
            cap = np.quantile(x, OVERFLOW_Q)
            if cap <= lo:
                cap = lo + (np.max(x) - lo) * 0.1 if np.max(x) > lo else lo + 1e-9
            binw = (cap - lo) / NBINS_MAIN
            m['lo'], m['cap'], m['binw'], m['edges_main'] = lo, cap, binw, np.linspace(lo, cap, NBINS_MAIN + 1)

        # ---------- Prepare stability metrics ----------
        stability_intervals = ['0_4', '4_10', '-5_10', '-5_30']
        stability_metrics = []
        
        # Collect ALL stability correlations across all intervals for shared x-axis PER COLUMN
        if stability_data is not None:
            for interval in stability_intervals:
                all_corrs = []
                for sess in session_numbers:
                    if sess in stability_data and interval in stability_data[sess]:
                        all_corrs.extend(stability_data[sess][interval])
                
                # Compute binning for THIS interval (column)
                if all_corrs:
                    x = np.asarray(all_corrs)
                    x = x[np.isfinite(x)]
                    if x.size > 0:
                        lo = np.min(x)
                        cap = np.quantile(x, OVERFLOW_Q)
                        if cap <= lo:
                            cap = lo + (np.max(x) - lo) * 0.1 if np.max(x) > lo else lo + 1e-9
                        binw = (cap - lo) / NBINS_MAIN
                        stability_metrics.append(dict(
                            key=f'__stability_{interval}__',
                            title=f'Stability {interval.replace("_", " to ")}s',
                            color='#3f0966',
                            xlabel='Odd-Even Corr',
                            global_values=np.array(all_corrs),
                            lo=lo, cap=cap, binw=binw,
                            edges_main=np.linspace(lo, cap, NBINS_MAIN + 1)
                        ))
                    else:
                        stability_metrics.append(dict(
                            key=f'__stability_{interval}__',
                            title=f'Stability {interval.replace("_", " to ")}s',
                            color='#3f0966',
                            xlabel='Odd-Even Corr',
                            global_values=np.array([]),
                            lo=0.0, cap=1.0, binw=1.0/NBINS_MAIN,
                            edges_main=np.linspace(0.0, 1.0, NBINS_MAIN + 1)
                        ))
                else:
                    stability_metrics.append(dict(
                        key=f'__stability_{interval}__',
                        title=f'Stability {interval.replace("_", " to ")}s',
                        color='#3f0966',
                        xlabel='Odd-Even Corr',
                        global_values=np.array([]),
                        lo=0.0, cap=1.0, binw=1.0/NBINS_MAIN,
                        edges_main=np.linspace(0.0, 1.0, NBINS_MAIN + 1)
                    ))

        # ---------- Compute peak widths ----------
        peak_widths_data = self._compute_peak_widths(long_position, psth_time)

        # ---------- Figure setup ----------
        n_cols_metrics = len(metrics)
        n_cols_stability = len(stability_metrics)
        n_cols_peak_width = 1  # One column for peak width
        n_cols_total = n_cols_metrics + n_cols_stability + n_cols_peak_width + (1 if INCLUDE_FOV else 0)
        fig_w = WIDTH_PER_COL * n_cols_total
        fig_h = 4.8 * (len(session_numbers) + 1)  # +1 for median row

        fig, axes = plt.subplots(len(session_numbers) + 1, n_cols_total, figsize=(fig_w, fig_h))
        if len(session_numbers) == 1:
            axes = np.array([axes])

        for j, m in enumerate(metrics):
            axes[0, j].set_title(m['title'], fontsize=30, fontweight='bold')
        
        # Add stability column titles
        for j, m in enumerate(stability_metrics):
            axes[0, n_cols_metrics + j].set_title(m['title'], fontsize=30, fontweight='bold')
        
        # Add peak width title
        axes[0, n_cols_metrics + n_cols_stability].set_title('Peak Width', fontsize=30, fontweight='bold')
        
        if INCLUDE_FOV:
            axes[0, -1].set_title('FOV', fontsize=30, fontweight='bold')

        # ---------- Per session ----------
        # Track values for median computation
        metric_values_per_col = {j: [] for j in range(n_cols_metrics)}
        stability_values_per_col = {j: [] for j in range(n_cols_stability)}
        peak_width_values = {'left': [], 'right': []}
        
        for i, session in enumerate(session_numbers):
            session_df = psth_data.xs(key=session, level='session')
            n_rois = len(session_df)
            
            # Plot original metrics
            for j, m in enumerate(metrics):
                vals = _peak_psth_series(session_df) if m['key'] == '__peak_psth__' else np.asarray(session_df[m['key']].values)
                vals = vals[np.isfinite(vals)]
                metric_values_per_col[j].extend(vals)
                
                lo, cap, binw, edges_main = m['lo'], m['cap'], m['binw'], m['edges_main']
                ax = axes[i, j]
                counts_main, _ = np.histogram(vals[vals <= cap], bins=edges_main)
                overflow_count = np.sum(vals > cap)
                centers_main = edges_main[:-1] + binw / 2
                center_over = edges_main[-1] + binw / 2
                ax.bar(centers_main, counts_main, width=binw, align='center',
                       edgecolor='black', color=m['color'])
                ax.bar(center_over, overflow_count, width=binw, align='center',
                       edgecolor='black', color=_darken_color(m['color'], OVERFLOW_DARKEN))
                if overflow_count > 0:
                    ax.text(center_over, overflow_count * 1.02, f'{overflow_count}',
                            ha='center', va='bottom', fontsize=20)
                ax.axvline(edges_main[-1], linestyle='--', color='gray', alpha=0.8, linewidth=1)
                ax.set_xlim(lo, edges_main[-1] + binw)
                if j in FORCE_ZERO_XLIM_COLS:
                    right = ax.get_xlim()[1]
                    ax.set_xlim(0, right)
                ax.set_title(f"{m['title']}\nMean={np.mean(vals):.3f}, Median={np.median(vals):.3f}",
                             fontsize=20, fontweight='bold')
                if j == 0:
                    ax.set_ylabel(f"Session {session}\n(n={n_rois})\nCount", fontsize=20, fontweight='bold')
                ax.set_xlabel(m['xlabel'], fontsize=20)
                ax.tick_params(axis='both', labelsize=14)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
            
            # Plot stability metrics
            for j, m in enumerate(stability_metrics):
                interval = m['key'].replace('__stability_', '').replace('__', '')
                if stability_data is not None and session in stability_data and interval in stability_data[session]:
                    vals = np.array(stability_data[session][interval])
                else:
                    vals = np.array([])
                
                vals = vals[np.isfinite(vals)]
                stability_values_per_col[j].extend(vals)
                
                lo, cap, binw, edges_main = m['lo'], m['cap'], m['binw'], m['edges_main']
                ax = axes[i, n_cols_metrics + j]
                
                if vals.size == 0:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                    ax.set_xlim(0, 1)
                    ax.set_ylim(0, 1)
                else:
                    counts_main, _ = np.histogram(vals[vals <= cap], bins=edges_main)
                    overflow_count = np.sum(vals > cap)
                    centers_main = edges_main[:-1] + binw / 2
                    center_over = edges_main[-1] + binw / 2
                    ax.bar(centers_main, counts_main, width=binw, align='center',
                           edgecolor='black', color=m['color'])
                    ax.bar(center_over, overflow_count, width=binw, align='center',
                           edgecolor='black', color=_darken_color(m['color'], OVERFLOW_DARKEN))
                    if overflow_count > 0:
                        ax.text(center_over, overflow_count * 1.02, f'{overflow_count}',
                                ha='center', va='bottom', fontsize=20)
                    ax.axvline(edges_main[-1], linestyle='--', color='gray', alpha=0.8, linewidth=1)
                    ax.set_xlim(lo, edges_main[-1] + binw)
                    ax.set_title(f"{m['title']}\nMean={np.mean(vals):.3f}, Median={np.median(vals):.3f}",
                                 fontsize=20, fontweight='bold')
                
                ax.set_xlabel(m['xlabel'], fontsize=20)
                ax.tick_params(axis='both', labelsize=14)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

            # Plot peak widths
            ax_pw = axes[i, n_cols_metrics + n_cols_stability]
            if session in peak_widths_data:
                widths_left = peak_widths_data[session]['left']
                widths_right = peak_widths_data[session]['right']
                
                peak_width_values['left'].extend(widths_left)
                peak_width_values['right'].extend(widths_right)
                
                all_widths = widths_left + widths_right
                if all_widths:
                    # Compute average of top 10 longest
                    top10_left = np.mean(sorted(widths_left, reverse=True)[:10]) if len(widths_left) >= 10 else np.mean(widths_left)
                    top10_right = np.mean(sorted(widths_right, reverse=True)[:10]) if len(widths_right) >= 10 else np.mean(widths_right)
                    
                    bins = np.linspace(0, max(all_widths), 40)
                    ax_pw.hist(widths_left, bins=bins, alpha=0.7, color='red', label='Left', edgecolor='black')
                    ax_pw.hist(widths_right, bins=bins, alpha=0.7, color='blue', label='Right', edgecolor='black')
                    ax_pw.legend(fontsize=14)
                    ax_pw.set_xlabel('Width (s)', fontsize=20)
                    ax_pw.set_title(f'Peak Width\nLeft top10: {top10_left:.2f}s, Right top10: {top10_right:.2f}s',
                                   fontsize=20, fontweight='bold')
                else:
                    ax_pw.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax_pw.transAxes)
            else:
                ax_pw.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax_pw.transAxes)
            
            ax_pw.tick_params(axis='both', labelsize=14)
            ax_pw.spines['top'].set_visible(False)
            ax_pw.spines['right'].set_visible(False)

            # Add FOV
            if INCLUDE_FOV:
                ax_img = axes[i, -1]
                img = None

                if len(plane):
                    img = plane.iloc[0]['mean_img']

                if isinstance(img, np.ndarray):
                    img = crop_sides(img)
                    lo_i, hi_i = np.percentile(img, 1), np.percentile(img, 98)
                    ax_img.imshow(img, cmap='gray', interpolation='lanczos', vmin=lo_i, vmax=hi_i)
                    ax_img.axis('off')
                else:
                    ax_img.text(0.5, 0.5, 'No image', ha='center', va='center')
                    ax_img.axis('off')

        # ---------- Add median line graph row ----------
        median_row = len(session_numbers)
        
        # Compute medians per session for each metric
        metric_medians_per_session = {j: [] for j in range(n_cols_metrics)}
        stability_medians_per_session = {j: [] for j in range(n_cols_stability)}
        peak_width_medians_per_session = {'left': [], 'right': []}
        
        for session in session_numbers:
            session_df = psth_data.xs(key=session, level='session')
            
            # Metrics
            for j, m in enumerate(metrics):
                vals = _peak_psth_series(session_df) if m['key'] == '__peak_psth__' else np.asarray(session_df[m['key']].values)
                vals = vals[np.isfinite(vals)]
                if vals.size > 0:
                    metric_medians_per_session[j].append(np.median(vals))
                else:
                    metric_medians_per_session[j].append(np.nan)
            
            # Stability
            for j, m in enumerate(stability_metrics):
                interval = m['key'].replace('__stability_', '').replace('__', '')
                if stability_data is not None and session in stability_data and interval in stability_data[session]:
                    vals = np.array(stability_data[session][interval])
                    vals = vals[np.isfinite(vals)]
                    if vals.size > 0:
                        stability_medians_per_session[j].append(np.median(vals))
                    else:
                        stability_medians_per_session[j].append(np.nan)
                else:
                    stability_medians_per_session[j].append(np.nan)
            
            # Peak widths
            if session in peak_widths_data:
                if peak_widths_data[session]['left']:
                    peak_width_medians_per_session['left'].append(np.median(peak_widths_data[session]['left']))
                else:
                    peak_width_medians_per_session['left'].append(np.nan)
                    
                if peak_widths_data[session]['right']:
                    peak_width_medians_per_session['right'].append(np.median(peak_widths_data[session]['right']))
                else:
                    peak_width_medians_per_session['right'].append(np.nan)
            else:
                peak_width_medians_per_session['left'].append(np.nan)
                peak_width_medians_per_session['right'].append(np.nan)
        
        # Plot median line graphs
        session_x = np.arange(len(session_numbers))
        
        # Metrics median line graphs
        for j, m in enumerate(metrics):
            ax = axes[median_row, j]
            medians = np.array(metric_medians_per_session[j])
            valid_mask = np.isfinite(medians)
            if np.any(valid_mask):
                ax.plot(session_x[valid_mask], medians[valid_mask], 'o-', 
                       color=m['color'], linewidth=2, markersize=8)
                ax.set_xlabel('Session', fontsize=16)
                ax.set_ylabel('Median', fontsize=16)
                ax.set_xticks(session_x)
                ax.set_xticklabels(session_numbers, fontsize=12)
                ax.tick_params(axis='both', labelsize=12)
                ax.grid(True, alpha=0.3)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
            else:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.axis('off')
        
        # Stability median line graphs
        for j, m in enumerate(stability_metrics):
            ax = axes[median_row, n_cols_metrics + j]
            medians = np.array(stability_medians_per_session[j])
            valid_mask = np.isfinite(medians)
            if np.any(valid_mask):
                ax.plot(session_x[valid_mask], medians[valid_mask], 'o-', 
                       color=m['color'], linewidth=2, markersize=8)
                ax.set_xlabel('Session', fontsize=16)
                ax.set_ylabel('Median', fontsize=16)
                ax.set_xticks(session_x)
                ax.set_xticklabels(session_numbers, fontsize=12)
                ax.tick_params(axis='both', labelsize=12)
                ax.grid(True, alpha=0.3)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
            else:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.axis('off')
        
        # Peak width median line graph
        ax = axes[median_row, n_cols_metrics + n_cols_stability]
        medians_left = np.array(peak_width_medians_per_session['left'])
        medians_right = np.array(peak_width_medians_per_session['right'])
        valid_mask_left = np.isfinite(medians_left)
        valid_mask_right = np.isfinite(medians_right)
        
        if np.any(valid_mask_left) or np.any(valid_mask_right):
            if np.any(valid_mask_left):
                ax.plot(session_x[valid_mask_left], medians_left[valid_mask_left], 'o-', 
                       color='red', linewidth=2, markersize=8, label='Left')
            if np.any(valid_mask_right):
                ax.plot(session_x[valid_mask_right], medians_right[valid_mask_right], 'o-', 
                       color='blue', linewidth=2, markersize=8, label='Right')
            ax.set_xlabel('Session', fontsize=16)
            ax.set_ylabel('Median Width (s)', fontsize=16)
            ax.set_xticks(session_x)
            ax.set_xticklabels(session_numbers, fontsize=12)
            ax.tick_params(axis='both', labelsize=12)
            ax.legend(fontsize=12)
            ax.grid(True, alpha=0.3)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
        
        # FOV median row
        if INCLUDE_FOV:
            axes[median_row, -1].axis('off')

        fig.suptitle(f"Subject {key['subject_id']} — Raw stats per session", fontsize=22, fontweight='bold')
        plt.tight_layout(rect=[0, 0.02, 1, 0.96])

        # ---------- Save ----------
        dir_root = (imgn.Parameters & dict(parameter_name='dir_root_save')).fetch1('parameter_value')[0]
        save_dir = Path(dir_root) / 'raw_stats'
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / f"subject_{key['subject_id']}_rawstats.png"
        fig.savefig(save_path, dpi=150)
        plt.close(fig)

        self.insert1({**key, 'raw_stats_path': str(save_path)})
        print(f"✅ Saved to {save_path}")

    def _compute_peak_widths(self, long_position, psth_time):
        """Compute peak width as time between half-amplitude around max."""
        peak_widths = defaultdict(lambda: {'left': [], 'right': []})
        
        for entry in long_position:
            sess = entry['session']
            
            for position in ['left', 'right']:
                key = f'psth_regular_long_{position}'
                if key in entry.dtype.names:
                    psth = entry[key]
                    if psth is not None and np.any(psth):
                        psth = np.array(psth).squeeze()
                        
                        # Find max and half-amplitude
                        max_val = np.max(psth)
                        half_amp = max_val / 2
                        max_idx = np.argmax(psth)
                        
                        # Find indices where signal crosses half-amplitude
                        above_half = psth >= half_amp
                        
                        # Find left crossing
                        left_idx = max_idx
                        for idx in range(max_idx, -1, -1):
                            if above_half[idx]:
                                left_idx = idx
                            else:
                                break
                        
                        # Find right crossing
                        right_idx = max_idx
                        for idx in range(max_idx, len(psth)):
                            if above_half[idx]:
                                right_idx = idx
                            else:
                                break
                        
                        # Convert to time
                        try:
                            if left_idx < right_idx:
                                width = psth_time[right_idx] - psth_time[left_idx]
                                peak_widths[sess][position].append(width)
                        except:
                            print(f"Error computing width for session {sess}, position {position}, indices {left_idx}-{right_idx}, length {len(psth_time)} length {len(psth)}.")
        return peak_widths

    def _compute_stability(self, long_position, midposition, short_position, psth_time):
        """Compute odd-even stability correlations for long condition."""
        
        conditions = ["long", "mid", "short"]
        fetch_all = {
            "short": short_position,
            "mid": midposition,
            "long": long_position
        }
        positions = ["right", "left"]
        rewards = ["large", "regular"]
        sigma = 10
        
        # Group by session
        fetch_by_session = {
            cond: defaultdict(lambda: defaultdict(list))
            for cond in conditions
        }
        
        for cond in conditions:
            for entry in fetch_all[cond]:
                sess = entry['session']
                subject = entry['subject_id']
                fetch_by_session[cond][sess][subject].append(entry)
        
        # Determine shortest PSTH length
        m = len(psth_time)
        
        # Initialize PSTH storage
        psths_zscore = {cond: defaultdict(lambda: defaultdict(dict)) for cond in conditions}
        
        for cond in conditions:
            for sess, subj_dict in fetch_by_session[cond].items():
                for subject, entries in subj_dict.items():
                    n = len(entries)
                    psths_zscore[cond][sess][subject] = {}
                    
                    # Preallocate arrays
                    for reward in rewards:
                        for position in positions:
                            base_key = f"{reward}_{cond}_{position}"
                            psths_zscore[cond][sess][subject][base_key] = np.zeros((n, m))
                            psths_zscore[cond][sess][subject][f"{reward}_odd_{cond}_{position}"] = np.zeros((n, m))
                            psths_zscore[cond][sess][subject][f"{reward}_even_{cond}_{position}"] = np.zeros((n, m))
                    
                    # Fill PSTHs
                    for i, entry in enumerate(entries):
                        for reward in rewards:
                            for position in positions:
                                for suffix in ["", "_odd", "_even"]:
                                    key_name = f"psth_{reward}{suffix}_{cond}_{position}"
                                    target_key = f"{reward}{suffix}_{cond}_{position}".replace("__", "_")
                                    
                                    if key_name in entry.dtype.names:
                                        data = entry[key_name]
                                        if data is not None and np.any(data):
                                            arr = np.array(data).squeeze()
                                            
                                            if len(arr) > m:
                                                arr = arr[:m]
                                            elif len(arr) < m:
                                                m = len(arr)
                                                for k in psths_zscore[cond][sess][subject]:
                                                    psths_zscore[cond][sess][subject][k] = psths_zscore[cond][sess][subject][k][:, :m]
                                                arr = arr[:m]
                                            
                                            psths_zscore[cond][sess][subject][target_key][i, :] = arr
        
        # Smooth all PSTHs
        for cond in conditions:
            for sess, subj_dict in psths_zscore[cond].items():
                for subject, psth_dict in subj_dict.items():
                    for key, arr in psth_dict.items():
                        psths_zscore[cond][sess][subject][key] = gaussian_filter1d(arr, sigma=sigma, axis=1)
        
        # Define time intervals
        stability_time_intervals = {
            "0_4": slice(np.searchsorted(psth_time, 0), np.searchsorted(psth_time, 4)),
            "4_10": slice(np.searchsorted(psth_time, 4), np.searchsorted(psth_time, 10)),
            "-5_10": slice(np.searchsorted(psth_time, -5), np.searchsorted(psth_time, 10)),
            "-5_30": slice(np.searchsorted(psth_time, -5), np.searchsorted(psth_time, 30))
        }
        
        # Compute correlations for long condition only
        concat_odd_even_corrs = defaultdict(lambda: defaultdict(list))
        
        cond = "long"
        for sess, subj_dict in psths_zscore[cond].items():
            for subject, psth_dict in subj_dict.items():
                n_entries = psth_dict[f"regular_odd_{cond}_left"].shape[0]
                
                for label, time_mask in stability_time_intervals.items():
                    for i in range(n_entries):
                        left_odd = psth_dict[f"regular_odd_{cond}_left"][i, time_mask]
                        right_odd = psth_dict[f"regular_odd_{cond}_right"][i, time_mask]
                        left_even = psth_dict[f"regular_even_{cond}_left"][i, time_mask]
                        right_even = psth_dict[f"regular_even_{cond}_right"][i, time_mask]
                        
                        odd_concat = np.concatenate([left_odd, right_odd])
                        even_concat = np.concatenate([left_even, right_even])
                        
                        if np.any(odd_concat) and np.any(even_concat):
                            corr = np.corrcoef(odd_concat, even_concat)[0, 1]
                            concat_odd_even_corrs[sess][label].append(corr)
        
        return concat_odd_even_corrs

In [ ]:
RawStats.populate()